# YellowPages Vietnam Deep Scraper — Demo Notebook

This notebook walks through the full usage of `YellowPagesScraper.py`:
1. Installation check
2. CLI usage examples
3. Python API usage
4. Inspecting the output CSV
5. Quick data analysis

## 1. Installation

Install required libraries (skip if already installed).

In [ ]:
!pip install requests beautifulsoup4 pandas --quiet

## 2. CLI Usage

The scraper is fully usable from the command line.
Run `--help` to see all available arguments.

In [3]:
!python YellowPagesScraper.py --help

usage: YellowPageScraper [-h] [--where LOCATION] [--max-pages N]
                         [--output FILE] [--delay-min SEC] [--delay-max SEC]
                         keyword

Scrape business data from yellowpages.vn

positional arguments:
  keyword               (Required) Industry / search keyword.
                          e.g. "May mac"

options:
  -h, --help            show this help message and exit
  --where LOCATION, -w LOCATION
                        (Optional) Target location.
                          e.g. "Binh Duong"  (default: nationwide)
  --max-pages N, -p N   (Optional) Max pages to scrape.
                          Default: auto-detect from div#paging.
  --output FILE, -o FILE
                        (Optional) Output CSV filename.
                          Default: results.csv
  --delay-min SEC       (Optional) Min delay between requests (seconds). Default: 2.0
  --delay-max SEC       (Optional) Max delay between requests (seconds). Default: 5.0

Examples:
  python 

### CLI Example — scrape 1 page, save to `results.csv`

```bash
python YellowPagesScraper.py "May mặc" --where "Bình Dương" --max-pages 1
```

In [5]:
!python YellowPagesScraper.py "May mặc" --where "Bình Dương" --max-pages 1

23:54:58  INFO        Fetching page 1 -> https://www.yellowpages.vn/srch/Binh_Dương/May_mặc.html
23:55:00  INFO      [PHASE 1] keyword='May mặc' | location='Binh Dương' | max_pages=1
23:55:00  INFO           -> Found 39 companies.
23:55:00  INFO      [PHASE 1] Done -- 39 unique URLs collected.
23:55:00  INFO      
[PHASE 2] Processing 39 companies ...
23:55:00  INFO        [   1/39] https://www.yellowpages.vn/lgs/1187759175/may-mac-tan-phuong-cong-ty-tnhh-may-mac-tan-phuong.html
23:55:06  INFO        [   2/39] https://www.yellowpages.vn/lgs/1187974575/mex-keo-dung-vai-one-brand-cong-ty-tnhh-san-pham-khong-det-new-one-brand-vn.html
23:55:12  INFO        [   3/39] https://www.yellowpages.vn/lgs/1187789079/giay-nganh-may-chong-am-cong-ty-tnhh-giay-cong-nghiep-binh-duong.html
23:55:17  INFO        [   4/39] https://www.yellowpages.vn/lgs/1187803290/giay-nganh-may-mac-da-giay-tien-hung--cong-ty-tnhh-sx-tm-xuat-nhap-khau-tien-hung.html
23:55:22  INFO        [   5/39] https://www.yellowpages.

## 3. Python API Usage

You can also import and use the scraper directly in your Python code.

In [ ]:
from YellowPagesScraper import YellowPagesScraper

scraper = YellowPagesScraper(
    keyword     = "May mac",
    where       = "Binh Duong",
    max_pages   = 1,          # set to None for auto-detect all pages
    delay_min   = 2.0,
    delay_max   = 4.0,
    output_file = "results.csv",
)

df = scraper.run()

11:42:01  INFO       Fetching page 1 -> https://www.yellowpages.vn/srch/Binh_Duong/May_mac.html
11:42:03  INFO       Detected div#paging -> total 7 page(s).
11:42:03  INFO     [PHASE 1] keyword='May mac' | location='Binh Duong' | max_pages=1
11:42:03  INFO          -> Found 20 companies.
11:42:03  INFO     [PHASE 1] Done -- 20 unique URLs collected.
11:42:03  INFO     [PHASE 2] Processing 20 companies ...
...
11:43:55  INFO     [DONE] Saved 20 records -> 'results.csv'


## 4. Output Preview

Load and inspect the generated CSV.

In [6]:
import pandas as pd

df = pd.read_csv("results.csv", encoding="utf-8-sig")
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print("\nColumns:", list(df.columns))

Shape: 38 rows x 13 columns

Columns: ['Company Name', 'Address', 'Phone', 'Website', 'Email', 'Tax Code', 'Business Type', 'Year Established', 'Target Market', 'Main Customers', 'No. of Employees', 'Industry', 'Detail URL']


In [ ]:
df.head()

In [8]:
# Check completeness of each column
completeness = (df.replace("", None).notna().sum() / len(df) * 100).round(1)
print("Field completeness (%):")
print(completeness.to_string())

Field completeness (%):
Company Name        100.0
Address             100.0
Phone               100.0
Website              94.7
Email               100.0
Tax Code             97.4
Business Type       100.0
Year Established     78.9
Target Market       100.0
Main Customers       57.9
No. of Employees     71.1
Industry            100.0
Detail URL          100.0


## 5. Quick Analysis

In [9]:
# Top business types
print("Business Type distribution:")
print(df["Business Type"].value_counts().head(5).to_string())

Business Type distribution:
Business Type
Nhà sản xuất                13
Nhà Sản Xuất                 7
Nhà sản xuất, Thương mại     3
Nhà sản xuất, Dịch vụ        2
Sản xuất                     2


In [10]:
# Year established distribution
print("Year Established distribution:")
print(df["Year Established"].value_counts().sort_index().to_string())

Year Established distribution:
Year Established
1992.0    1
2002.0    2
2003.0    1
2004.0    1
2005.0    1
2006.0    3
2007.0    1
2008.0    3
2010.0    2
2012.0    1
2013.0    1
2015.0    2
2016.0    1
2017.0    2
2018.0    2
2019.0    2
2020.0    1
2021.0    1
2022.0    1
2025.0    1


In [11]:
# Export a filtered subset: only companies with a website
df_with_website = df[df["Website"].str.strip() != ""]
print(f"Companies with website: {len(df_with_website)}/{len(df)}")
df_with_website.to_csv("results_with_website.csv", index=False, encoding="utf-8-sig")
print("Saved -> results_with_website.csv")

Companies with website: 38/38
Saved -> results_with_website.csv


---
## Summary

| Step | What happened |
|---|---|
| Phase 1 | Fetched 1 search page, found 20 company URLs |
| Phase 2 | Visited each URL, extracted 13 fields per company |
| Output | `results.csv` — utf-8-sig encoded, ready for Excel |

**Next steps:**
- Remove `max_pages=1` to scrape all 7 pages (~140 companies)
- Change `keyword` and `where` to target other industries / cities